# AEV-PLIG Results Analysis
Multi-model accuracy · ensemble agreement · uncertainty calibration · ranking · outliers

In [ ]:
import re

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import erf
from scipy.stats import kendalltau as scipy_kendalltau

from aev_plig import results

In [ ]:
PREDICTIONS_DIR      = "output/predictions"
# Stem of the parquet file produced by scripts/predict.py, e.g.:
# "pdbbind_U_bindingnet_U_bindingdb_ligsim90_fep_benchmark_predictions"
PREDICTION_FILE_NAME = "fep_benchmark_predictions"
TRUTH_COL            = "pK"
PRED_COL             = "preds"
UID_COL              = "unique_id"
TOP_N_OUTLIERS       = 20
MIN_TARGET_SAMPLES   = 2
FIG_DIR              = None   # set to a Path to auto-save HTML figures

In [ ]:
df = results.load_all_predictions(PREDICTIONS_DIR, prediction_file_name=PREDICTION_FILE_NAME)

# Auto-detect ensemble member columns (preds_0, preds_1, ...)
pred_member_cols = sorted(
    [c for c in df.columns if re.fullmatch(r"preds_\d+", c)],
    key=lambda c: int(c.split("_")[1]),
)
n_models = len(pred_member_cols)

# Detect Bayesian output (var_0 ... var_N present)
var_member_cols = sorted(
    [c for c in df.columns if re.fullmatch(r"var_\d+", c)],
    key=lambda c: int(c.split("_")[1]),
)
is_bayesian_output = bool(var_member_cols)

print(
    f"Rows: {df.height:,} | ensemble members: {n_models} "
    f"| Bayesian output: {is_bayesian_output}"
)

# Epistemic std = disagreement across the N ensemble checkpoints
df = df.with_columns(
    (
        pl.concat_list([pl.col(c) for c in pred_member_cols]).list.std()
        if n_models > 1
        else pl.lit(0.0)
    ).alias("epistemic_std")
)

# Aleatoric std = sqrt(mean aleatoric variance from Bayesian head), pK units
if is_bayesian_output:
    df = df.with_columns(pl.col("var").sqrt().alias("aleatoric_std"))

# Residual (signed: predicted - true)
if TRUTH_COL in df.columns:
    df = df.with_columns((pl.col(PRED_COL) - pl.col(TRUTH_COL)).alias("residual"))

df.head(3)

## §1 Data Overview

In [ ]:
clean = df.filter(pl.col(TRUTH_COL).is_not_null())
color_col = "model_name" if "model_name" in clean.columns else None
fig = px.histogram(
    clean.to_dict(as_series=False),
    x=TRUTH_COL,
    nbins=40,
    color=color_col,
    barmode="overlay",
    opacity=0.7,
    title=f"Distribution of true {TRUTH_COL}  (n={clean.height:,})",
)
fig.show()
print(clean.select(TRUTH_COL).describe())

## §2 Ensemble Member Metrics
RMSE, Pearson R, and Kendall τ for every ensemble member and the ensemble average,
grouped by `model_instance` when multiple model runs are loaded.

In [ ]:
id_cols = [TRUTH_COL] + (["model_instance"] if "model_instance" in df.columns else [])
long = (
    df.select(id_cols + pred_member_cols + [PRED_COL])
    .drop_nulls()
    .unpivot(
        on=pred_member_cols + [PRED_COL],
        index=id_cols,
        variable_name="model_col",
        value_name="pred",
    )
)

group_keys = ["model_instance", "model_col"] if "model_instance" in long.columns else ["model_col"]

metrics_df = (
    long.group_by(group_keys, maintain_order=True)
    .agg(
        ((pl.col("pred") - pl.col(TRUTH_COL)).pow(2).mean().sqrt()).alias("RMSE"),
        pl.pearson_corr("pred", TRUTH_COL).alias("Pearson R"),
        pl.map_groups(
            ["pred", TRUTH_COL],
            lambda s: float(scipy_kendalltau(s[0].to_numpy(), s[1].to_numpy()).statistic),
        ).alias("Kendall τ"),
    )
    .sort(group_keys)
)
display(metrics_df)

fig = px.bar(
    metrics_df.to_dict(as_series=False),
    x="model_col",
    y="RMSE",
    color="model_instance" if "model_instance" in metrics_df.columns else None,
    barmode="group",
    title="RMSE per ensemble member vs ensemble average",
    labels={"model_col": "Model"},
)
fig.show()

## §3 Predicted vs True
Colour and error bars show epistemic std (ensemble disagreement).

In [ ]:
scatter_cols = (
    [TRUTH_COL, PRED_COL, UID_COL, "epistemic_std"]
    + (["model_instance"] if "model_instance" in df.columns else [])
)
plot_df = df.select(scatter_cols).drop_nulls()
x_range = [float(plot_df[TRUTH_COL].min()), float(plot_df[TRUTH_COL].max())]

fig = px.scatter(
    plot_df.to_dict(as_series=False),
    x=TRUTH_COL,
    y=PRED_COL,
    color="epistemic_std",
    color_continuous_scale="Viridis",
    error_y="epistemic_std",
    facet_col="model_instance" if "model_instance" in plot_df.columns else None,
    hover_data=[UID_COL],
    title="Predicted vs True  (colour/error bars = epistemic std dev)",
)
n_facets = plot_df["model_instance"].n_unique() if "model_instance" in plot_df.columns else 1
for _ in range(n_facets):
    fig.add_trace(
        go.Scatter(
            x=x_range, y=x_range, mode="lines",
            line=dict(dash="dash", color="black"), showlegend=False,
        )
    )
fig.show()

## §4 Residual Analysis

In [ ]:
res = df.select([TRUTH_COL, "residual", UID_COL]).drop_nulls()
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Residual distribution", f"Residual vs True {TRUTH_COL}"],
)
fig.add_trace(
    go.Histogram(x=res["residual"].to_numpy(), nbinsx=40, name="Residuals"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=res[TRUTH_COL].to_numpy(), y=res["residual"].to_numpy(),
        mode="markers", text=res[UID_COL].to_list(),
        marker=dict(opacity=0.5), name="Residual",
    ),
    row=1, col=2,
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(title="Residual Analysis", showlegend=False)
fig.show()

## §5 Uncertainty Calibration

**Epistemic std** = std dev of the N ensemble checkpoint predictions (`std(preds_0…N)`).
Valid for all model types.

**Aleatoric std** *(Bayesian models only)* = `sqrt(mean(var_0…N))`, where each `var_i`
is the variance from the model’s `logvar_head` (softplus-activated, denormalised to pK²).
This is the model’s self-reported data noise, independent of ensemble disagreement.

Three diagnostics are shown for each uncertainty source:
1. **Scatter** — uncertainty vs |residual|; Pearson R quantifies linear correlation
2. **Reliability diagram** — bin by uncertainty quantile, plot mean |error| per bin; a
   well-calibrated model has a monotonically increasing curve
3. **Sparsification curve** — rank predictions from most to least confident; progressively
   include more uncertain predictions and track RMSE; good calibration shows RMSE rising
   as uncertain predictions are added
4. **Prediction interval coverage** — what fraction of true values fall within ±kσ;
   a Gaussian-calibrated model matches the theoretical erf curve

In [ ]:
def reliability_diagram(unc: np.ndarray, abs_err: np.ndarray, n_bins: int = 10):
    """Equal-frequency bins; return (mean_unc, mean_abs_err) per bin. Vectorised."""
    bin_edges = np.percentile(unc, np.linspace(0, 100, n_bins + 1))
    bin_idx   = np.searchsorted(bin_edges[1:-1], unc)  # 0 … n_bins-1
    counts    = np.bincount(bin_idx, minlength=n_bins).astype(float)
    mean_unc  = np.bincount(bin_idx, weights=unc,     minlength=n_bins) / counts
    mean_err  = np.bincount(bin_idx, weights=abs_err, minlength=n_bins) / counts
    return mean_unc, mean_err


def sparsification_curve(unc: np.ndarray, residuals: np.ndarray):
    """Sort by unc (most confident first); return cumulative RMSE as uncertain preds are added."""
    order         = np.argsort(unc)
    sq_err_sorted = residuals[order] ** 2
    cum_rmse      = np.sqrt(np.cumsum(sq_err_sorted) / np.arange(1, len(sq_err_sorted) + 1))
    frac_retained = np.arange(1, len(sq_err_sorted) + 1) / len(sq_err_sorted)
    return frac_retained, cum_rmse


def interval_coverage(unc: np.ndarray, residuals: np.ndarray):
    """Observed vs expected (Gaussian) coverage at k = 0.5, 1, 1.5, 2 sigma."""
    k_vals       = np.array([0.5, 1.0, 1.5, 2.0])
    obs_cov      = (np.abs(residuals)[None, :] <= k_vals[:, None] * unc[None, :]).mean(axis=1)
    expected_cov = erf(k_vals / np.sqrt(2))
    return k_vals, obs_cov, expected_cov


def calibration_panel(unc: np.ndarray, residuals: np.ndarray, label: str):
    abs_err = np.abs(residuals)
    r       = float(np.corrcoef(unc, abs_err)[0, 1])

    # 1 + 2: scatter and reliability diagram
    mu_unc, mu_err = reliability_diagram(unc, abs_err)
    fig1 = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f"Scatter (Pearson R = {r:.3f})", "Reliability diagram"],
    )
    xs = np.sort(unc)
    fig1.add_trace(go.Scatter(x=unc, y=abs_err, mode="markers",
                              marker=dict(opacity=0.25, size=4), name="Points"), row=1, col=1)
    fig1.add_trace(go.Scatter(x=xs, y=np.poly1d(np.polyfit(unc, abs_err, 1))(xs),
                              mode="lines", line=dict(color="red"), name="Fit"), row=1, col=1)
    fig1.add_trace(go.Scatter(x=mu_unc, y=mu_err, mode="lines+markers",
                              marker=dict(size=8), name="Mean per bin"), row=1, col=2)
    fig1.update_xaxes(title_text=f"{label} (pK)", row=1, col=1)
    fig1.update_xaxes(title_text=f"Mean {label} per bin (pK)", row=1, col=2)
    fig1.update_yaxes(title_text="|Residual| (pK)")
    fig1.update_layout(title=f"Calibration — {label}", showlegend=False)
    fig1.show()

    # 3: sparsification curve
    frac, cum_rmse = sparsification_curve(unc, residuals)
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=frac, y=cum_rmse, mode="lines", name="Model"))
    fig2.update_layout(
        title=f"Sparsification — {label}",
        xaxis_title="Fraction retained (most confident first)",
        yaxis_title="Cumulative RMSE (pK)",
    )
    fig2.show()

    # 4: prediction interval coverage
    k_vals, obs_cov, expected_cov = interval_coverage(unc, residuals)
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=k_vals, y=obs_cov, mode="lines+markers",
                              name="Observed"))
    fig3.add_trace(go.Scatter(x=k_vals, y=expected_cov, mode="lines",
                              line=dict(dash="dash", color="gray"), name="Gaussian (ideal)"))
    fig3.update_layout(
        title=f"Prediction interval coverage — {label}",
        xaxis_title="k (±kσ)",
        yaxis_title="Coverage",
        yaxis=dict(range=[0, 1.05]),
    )
    fig3.show()


if n_models > 1:
    cal_e = df.select(["epistemic_std", "residual"]).drop_nulls()
    calibration_panel(
        cal_e["epistemic_std"].to_numpy(),
        cal_e["residual"].to_numpy(),
        "Epistemic std (ensemble)",
    )
    if is_bayesian_output:
        cal_a = df.select(["aleatoric_std", "residual"]).drop_nulls()
        calibration_panel(
            cal_a["aleatoric_std"].to_numpy(),
            cal_a["residual"].to_numpy(),
            "Aleatoric std (Bayesian head)",
        )
else:
    print("Single model — uncertainty calibration requires multiple ensemble members.")

## §6 Per-Target Kendall τ (Ranking Ability)
Each `unique_id` is treated as its own target.

In [ ]:
target_metrics = results.per_target_metrics(
    df,
    target_col=UID_COL,
    truth_col=TRUTH_COL,
    pred_col=PRED_COL,
    min_samples=MIN_TARGET_SAMPLES,
)
fig = px.histogram(
    target_metrics.to_dict(as_series=False),
    x="kendall_tau",
    nbins=30,
    title="Per-target Kendall τ distribution",
    labels={"kendall_tau": "Kendall τ"},
)
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.show()
display(target_metrics.sort("kendall_tau"))

## §7 Outlier Table

In [ ]:
outlier_cols = (
    [UID_COL, TRUTH_COL, PRED_COL, "residual", "epistemic_std"]
    + (["aleatoric_std"] if is_bayesian_output else [])
    + (["model_instance"] if "model_instance" in df.columns else [])
)
outliers = (
    df.select(outlier_cols)
    .drop_nulls()
    .with_columns(pl.col("residual").abs().alias("abs_residual"))
    .sort("abs_residual", descending=True)
    .head(TOP_N_OUTLIERS)
)
display(outliers)

## §8 Success Rate

In [ ]:
thresholds = pl.Series([0.5, 1.0, 1.5, 2.0])
abs_res    = df.select("residual").drop_nulls()["residual"].abs()
success = pl.DataFrame({
    "Threshold (±pK)": thresholds,
    "N within": thresholds.map_elements(
        lambda t: int((abs_res <= t).sum()), return_dtype=pl.Int64
    ),
    "% within": thresholds.map_elements(
        lambda t: round(100.0 * float((abs_res <= t).mean()), 1), return_dtype=pl.Float64
    ),
})
display(success)